
 ### <font color="orange">🧠 GIAI ĐOẠN 3: KỶ NGUYÊN LLMs</font>
 #### Bài 28: Kỹ năng "Bịt mắt đoán chữ" (Causal Masking)

 **1. Tại sao phải Bịt mắt?**
 GPT là một mô hình sinh chữ (Generative). Nó học bằng cách đọc chữ thứ 1, đoán chữ thứ 2. Đọc chữ thứ 1+2, đoán chữ thứ 3.
 Nếu trong lúc train, em để lộ toàn bộ câu cho nó thấy (như BERT), nó sẽ "gian lận" bằng cách nhìn đáp án ở tương lai thay vì tự suy luận.

 **2. Causal Masking (Mặt nạ thời gian) hoạt động thế nào?**
 Hãy tưởng tượng ma trận điểm số Attention (Scores) là một bảng vuông (Kích thước: số từ x số từ).
 Hàng 1 (Từ thứ 1): Chỉ được nhìn chính nó. Các từ thứ 2, 3, 4 phải bị che lại.
 Hàng 2 (Từ thứ 2): Chỉ được nhìn từ 1 và 2. Các từ 3, 4 phải bị che lại.

 -> Giải pháp Toán học: Ta biến tất cả các ô nằm "trên đường chéo chính" (tương lai) thành **âm vô cùng ($-\infty$)**.
 Khi đi qua hàm $Softmax$, $e^{-\infty} = 0$. Mức độ chú ý vào tương lai sẽ chính xác bằng 0%!



In [1]:
import torch
import torch.nn.functional as F

def main():
    print("--- BÀI 28: CHẾ TẠO MẶT NẠ THỜI GIAN CHO GPT ---")
    torch.manual_seed(42)
    
    seq_len = 5 # Giả sử câu văn có 5 từ
    
    # Giả lập ma trận điểm số Attention Scores (Q * K^T / sqrt(d_k))
    # Kích thước 5x5
    scores = torch.randn(seq_len, seq_len)
    
    print("1. Ma trận Scores GỐC (Chưa bịt mắt):")
    print(torch.round(scores * 100) / 100) # Làm tròn cho dễ nhìn
    
    # ========================================================
    # NHIỆM VỤ CỦA EM: TẠO MẶT NẠ VÀ BỊT MẮT TƯƠNG LAI
    # ========================================================
    
    # Bước 1: Tạo ma trận tam giác dưới (Lower Triangular) toàn số 1.
    # Phần trên đường chéo sẽ là số 0.
    # Gợi ý PyTorch: torch.tril(torch.ones(seq_len, seq_len))
    
    # TỰ CODE VÀO ĐÂY:
    mask = torch.tril(torch.ones(seq_len, seq_len))
    
    # Bước 2: Dùng mặt nạ để che điểm số.
    # Lệnh `scores.masked_fill(điều_kiện, giá_trị)` sẽ tìm những chỗ 
    # khớp với điều kiện và thay nó bằng một giá trị khác.
    # Yêu cầu: Hãy thay những chỗ mask == 0 bằng âm vô cùng: float('-inf')
    
    # TỰ CODE VÀO ĐÂY:
    masked_scores = scores.masked_fill(mask == 0, float('-inf')) # Sửa lại dòng này!
    
    # Bước 3: Đưa qua hàm Softmax để biến thành phần trăm (0-100%)
    attention_weights = F.softmax(masked_scores, dim=-1)
    
    try:
        print("\n2. Mặt nạ (Mask) em vừa tạo:")
        print(mask)
        
        print("\n3. Ma trận Scores sau khi bịt mắt bằng -inf:")
        print(masked_scores)
        
        print("\n4. Trọng số Attention cuối cùng (Đã qua Softmax):")
        print(torch.round(attention_weights * 1000) / 1000)
        
        # Kiểm tra dòng đầu tiên (Từ thứ 1) xem tổng các từ tương lai có bằng 0 không
        if attention_weights[0, 1:].sum() == 0:
            print("\n✅ THÀNH CÔNG! Từ thứ 1 đã bị mù hoàn toàn với tương lai (từ 2,3,4,5 đều bằng 0%)")
            print("💡 GPT đã sẵn sàng để tự học mà không lo gian lận!")
            
    except Exception as e:
        print(f"\n❌ Có lỗi: {e}")

if __name__ == "__main__":
    main()

--- BÀI 28: CHẾ TẠO MẶT NẠ THỜI GIAN CHO GPT ---
1. Ma trận Scores GỐC (Chưa bịt mắt):
tensor([[ 1.9300,  1.4900,  0.9000, -2.1100,  0.6800],
        [-1.2300, -0.0400, -1.6000, -0.7500, -0.6900],
        [-0.4900,  0.2400, -1.1100,  0.0900, -2.3200],
        [-0.2200, -1.3800, -0.4000,  0.8000, -0.6200],
        [-0.5900, -0.0600, -0.8300,  0.3300, -1.5600]])

2. Mặt nạ (Mask) em vừa tạo:
tensor([[1., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0.],
        [1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1.]])

3. Ma trận Scores sau khi bịt mắt bằng -inf:
tensor([[ 1.9269,    -inf,    -inf,    -inf,    -inf],
        [-1.2345, -0.0431,    -inf,    -inf,    -inf],
        [-0.4934,  0.2415, -1.1109,    -inf,    -inf],
        [-0.2168, -1.3847, -0.3957,  0.8034,    -inf],
        [-0.5920, -0.0631, -0.8286,  0.3309, -1.5576]])

4. Trọng số Attention cuối cùng (Đã qua Softmax):
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2330, 0.7670, 0.0000, 